## Классификация авиакатастроф

In [54]:
import spacy
from spacy.matcher import Matcher
from spacy.util import filter_spans

nlp = spacy.load("ru_core_news_lg")

matcher = Matcher(nlp.vocab)

# --- списки слов ---
aircraft_terms = [
    "самол[её]т", "авиалайнер", "вертол[её]т",
    "ту-", "ан-", "ил-", "ми-", "миг-", "cy-", "superjet", "боинг", "airbus",
]
crash_verbs = ["разбиться", "рухнуть", "упасть", "потерпеть"]

# === глагольные случаи (самолет разбился) ===
pattern_verb = [
    {"LOWER": {"REGEX": "|".join(aircraft_terms)}},
    {"IS_ASCII": True, "OP": "*"},
    {"LEMMA": {"IN": crash_verbs}}
]
pattern_verb_reversed = [
    {"LEMMA": {"IN": crash_verbs}},
    {"IS_ASCII": True, "OP": "*"},
    {"LOWER": {"REGEX": "|".join(aircraft_terms)}}
]
matcher.add("PLANE_CRASH_EVENT", [pattern_verb, pattern_verb_reversed])

# === Само по себе слово авиакатастрофа автоматически сигнализирует о матче ===
plane_crash_pattern = [[{"LEMMA": {"IN": ["авиакатастрофа"]}}]]
matcher.add("CRASH_EVENT", plane_crash_pattern)

# === существительные случаи (авиакатастрофа) ===
crash_nouns = ["крушение", "столкновение", "авария", "катастрофа"]
pattern_noun = [
    {"LEMMA": {"IN": crash_nouns}},
    {"IS_ASCII": True, "OP": "*"},
    {"LOWER": {"REGEX": "|".join(aircraft_terms)}}
]
pattern_noun_reversed = [
    {"LOWER": {"REGEX": "|".join(aircraft_terms)}},
    {"IS_ASCII": True, "OP": "*"},
    {"LEMMA": {"IN": crash_nouns}}
]
matcher.add("PLANE_CRASH_EVENT", [pattern_noun, pattern_noun_reversed])

# === отдельного внимания заслуживает аварийная посадка
pattern_emergency = [
    {"LEMMA": "аварийный"},
    {"LEMMA": "посадка"},
    {"LOWER": {"REGEX": "|".join(aircraft_terms)}, "OP": "?"}
]
matcher.add("PLANE_CRASH_EVENT", [pattern_emergency])

# === пример текста ===
text = """
Крушение авиалайнера «Saab 340» авиакомпании «Crossair» у деревни Нассенвиль.
Самолёт потерпел крушение в районе вулкана Опала на Камчатке.
Рухнула стена склада в Петербурге.
Вертолёт Ми-8 разбился при взлёте из аэропорта города Таманрассет.
SuperJet совершил аварийную посадку.
Авиакатастрофа в индонезии. Самолет superjet 100.
Произошла аварийная посадка самолета.
Крушение самолета в Израиле.
Самолет потерпел крушение в Израиле.
Самолёт «Fokker F28-1000 Fellowship» перуанской авиакомпании TANS Perú потерпел катастрофу на подлёте к аэропорту вблизи Чачапояс
"""
doc = nlp(text)

# === поиск ===
matches = matcher(doc)

# --- собираем спаны с метками ---
spans = []
for match_id, start, end in matches:
    label = nlp.vocab.strings[match_id]
    span = doc[start:end]
    spans.append((label, span))

# --- убираем дубликаты и вложенные спаны ---
unique_spans = filter_spans([s for _, s in spans])

final_spans = []
for sent in doc.sents:
    sent_spans = [s for s in unique_spans if s.start >= sent.start and s.end <= sent.end]
    if sent_spans:
        # берём самый длинный (по количеству токенов)
        longest = max(sent_spans, key=lambda s: s.end - s.start)
        final_spans.append(longest)

# --- вывод ---
for i, span in enumerate(final_spans, start=1):
    sent = span.sent.text.strip()
    label = [lbl for lbl, s in spans if s.start == span.start and s.end == span.end][0]
    print(f"▶ {i} {sent}")
    print(f"  ↳ match: «{span.text}» → {label}\n")

▶ 1 Крушение авиалайнера «Saab 340» авиакомпании «Crossair» у деревни Нассенвиль.
  ↳ match: «Крушение авиалайнера» → PLANE_CRASH_EVENT

▶ 2 Самолёт потерпел крушение в районе вулкана Опала на Камчатке.
  ↳ match: «Самолёт потерпел» → PLANE_CRASH_EVENT

▶ 3 Вертолёт Ми-8 разбился при взлёте из аэропорта города Таманрассет.
  ↳ match: «Ми-8 разбился» → PLANE_CRASH_EVENT

▶ 4 SuperJet совершил аварийную посадку.
  ↳ match: «аварийную посадку» → PLANE_CRASH_EVENT

▶ 5 Авиакатастрофа в индонезии.
  ↳ match: «Авиакатастрофа» → CRASH_EVENT

▶ 6 Самолет superjet 100.
Произошла аварийная посадка самолета.
  ↳ match: «аварийная посадка» → PLANE_CRASH_EVENT

▶ 7 Самолет потерпел крушение в Израиле.
  ↳ match: «Самолет потерпел» → PLANE_CRASH_EVENT



In [55]:
def is_plane_crash(text, nlp=nlp, matcher=matcher):
    return bool(matcher(nlp(text)))


examples = [
    "Самолёт потерпел крушение в Камчатке.",
    "Пилоты Boeing 737 успешно совершили посадку.",
    "Вертолёт Ми-8 разбился в Алтае.",
    "Компания Airbus представила новый лайнер.",
]

for t in examples:
    print(f"{is_plane_crash(t)} → {t}")

True → Самолёт потерпел крушение в Камчатке.
False → Пилоты Boeing 737 успешно совершили посадку.
True → Вертолёт Ми-8 разбился в Алтае.
False → Компания Airbus представила новый лайнер.


In [56]:
import pandas as pd

df = pd.read_csv('../events/2_struct/2000-2025.csv')
df = df.sample(random_state=42, n=100)

df['plane_crash'] = df['event'].apply(is_plane_crash)

df.head()

,date_start,date_end,event,plane_crash
3135,2014-06-11,NaN,Луганский аэропорт прекратил любую деятельност...,False
1835,2008-06-21,NaN,Победа сборной России в четвертьфинале чемпион...,False
3832,2017-08-04,2017-08-13,"чемпионат мира по лёгкой атлетике (Лондон, Вел...",False
1919,2009-01-18,NaN,Израиль в одностороннем порядке прекратил боев...,False
4418,2021-01-01,NaN,Начало поставки компанией «Газпром» природного...,False


In [57]:
df = pd.read_csv('../events/2_struct/2000-2025.csv')
df['plane_crash'] = df['event'].apply(is_plane_crash)
df = df[df['plane_crash'] == True]

# todo распарсить кол-во смертей в авиакатастрофах
# отдельно рассмотреть авиакатастрофы с участием высокопоставленных лиц
# todo посмотреть, все ли я нашел с помощью своих правил
df

,date_start,date_end,event,plane_crash
10,2000-01-10,NaN,Крушение авиалайнера «Saab 340» авиакомпании «...,True
98,2000-04-19,NaN,На острове Самар при заходе на посадку в аэроп...,True
167,2000-07-04,NaN,"авария Ту-154 в Салониках, никто из находивших...",True
246,2000-10-25,NaN,"катастрофа Ил-18 под Батуми, погибли 84 человека.",True
415,2001-06-22,NaN,в Воронежской области при выполнении авиационн...,True
...,...,...,...,...
5413,2025-01-29,NaN,Авиакатастрофа в Вашингтоне из-за столкновения...,True
5459,2025-02-25,NaN,самолёт ВВС Судана Ан-26 разбился в жилом райо...,True
5560,2025-06-12,NaN,крушение Boeing 787 под Ахмадабадом в Индии. С...,True
5585,2025-07-21,NaN,в Бангладеш военный самолёт упал на территорию...,True
